In [4]:
from whitebox_workflows import WbEnvironment
from whitebox_workflows import Raster
import geopandas
import pandas as pd
import math

In [2]:
wbe = WbEnvironment()

Streamnetwork Convert to Raster

In [5]:
stream_shapefile = r"C:\Work\imWEBs\BRC\model\BRC\Watershed\input\stream_network.shp"
dem_rasterfile = r"C:\Work\imWEBs\BRC\model\BRC\Watershed\input\DEM.tif"

dem_raster = wbe.read_raster(dem_rasterfile)
stream_vector = wbe.read_vector(stream_shapefile)
stream_raster = wbe.vector_lines_to_raster(input=stream_vector, base_raster=dem_raster)

In [6]:
stream_raster

In [7]:
wbe.write_raster(stream_raster, r"C:\Work\imWEBs\BRC\model\BRC\Watershed\test\stream.tif")

Create Wetland Type Raster - createDifferentWetlandTypeMap. It seems the result raster is not used later. Do we need it?

There are two wetland rasters
1. Raster with wetland id. This is created based on wetland id.
2. Raster with wetland type id. This is created based on wetland type. Should be just another conversion with different column.
    Altered: 2
    Intact: 1
    DrainedAltered: 3
    DrainedConsolidated: 4
    DrainLost: 5

In [10]:
def createDifferentWetlandType(wetland_shapefile:str, wetland_type_field:str, wetland_raster:str, wetland_class_raster:str):
    """crate wetland type raster"""
    pass

**Burn Structure BMP**

Step 1: Clip the DEM - Watershed.getClippedDEM - **DONE**

1. If user has no watershed boundary, create the mask raster by overlaying dem, landuse and soil (wbe.multiply_overlay or use raster operators). The assumption is that all the rasters having the same size. And then apply the mask to the dem (raster operator).
2. If uses provides the boundary, clip the dem using the boundary shapefile (wbe.clip_raster_to_polygon)

Step 2: If user provides a stream shapefile, burn Stream (wbe.fill_burn) - **DONE**

Step 3: Fill Depression (wbe.fill_depressions) - **DONE**

Step 4: Filter Wetland (watershed.filterWetland) - find different type of wetlands including ones with area smaller than a threshold value - **DONE, NOT REQUIRED ANY MORE**

    Inputs:
    1. wetland raster - getWetlandRaster
    2. wetland outlets - getWetlandOutletsRaster
    3. main stream
    4. threashold area

    Outputs:

    1. wetlandNoSmall - ones without small ones - getWetlandMdfyRaster
    2. wetlandNoSmallMulti - ones with multiple outlets - getWetlandIsoMOutRaster
    3. wetlandIsolated - ones not connected to main stream directly - getWetlandIsolateRaster
    4. wetlandRip - ones connected to main stream - getWetlandRipaOutRaster




Step 5: Create wetland polygons from getWetlandMdfyRaster - ""DONE**

Step 6: Wetland Outlets Processing. This also applies to Feedlot, Dugout, Catchbasin and Manure Storage. For Catchbasin and Manure Storage, user usually doesn't provided outlets so just call WetlandOutletsPFO. For all the structure bmps except wetland, isAllowMultiOutlet is set to False and isUsingPFO is set to True.

1. If users provide the outlet, offset the outlets to snap to wetland (customized plugin WetlandOutletsOffset) - **DONE**
2. If users doesn't provide the outlets, generate the outlets (watershed.buildOutletStructureBMP, customized plugin WetlandOutletsPFO). If isUsingPFO == False, run FlowPointerD8 (wbe.d8_pointer) and use result as input for the plugin. - **DONE with Zhangbin's Function which generates the outlets and modify the wetland**

Step 7: Combine outlets of wetland and feedlots and burn to flow direction (Watershed.burnFlowDirection) **NEED DISCUSSION**

1. Combine the outlets of wetland, feedlots to single raster
2. Combine wetland and feedlot polygons to a single shapefile and convert to raster. 
3. Call customized plugin (WetlandDirection) to burn to the flow direction. This is a very long plugin.

**Watershed Outlets**

Watershed.delineatePourPoints

1. FlowAccumD8 - **Just call wbe.d8_flow_accum with modified flow direction**
2. MaxUpslopeFlowpathLength - **DONE**
3. ExtractStreams - **Call web.extract_streams with threshold. Note the threshold is the number of cells**
4. RasterStreamsToVector - MainStream and StreamNetwork - **wbe.raster_streams_to_vector**
5. buildPourPoints with wetland, Feedlot and Dugout - Watershed.buildPourPoints - **DONE**
6. User provided oultet shapefile
    a. JensonSnapPourPoints - Snap pour points - **wbe.jenson_snap_pour_points**
    b. VectorPointsToRaster - Convert Jenson vector file to raster - **wbe.vector_points_to_raster**
7. Reservoir 
    a. JensonSnapPourPoints - Snap pour points - **wbe.jenson_snap_pour_points**
    b. VectorPointsToRaster - Convert Jenson vector file to raster - **wbe.vector_points_to_raster**
8. MergeShapefiles - Merge outlets from three sources: buildPourpoints, user provided outlets, reservoir - **wbe.merge_vectors**
9. Set FID in the merged shapefile starting from 1, i.e. re-id the outlets. - **use geopandas**
10. VectorPointsToRaster - Convert MergeShapefiles to raster - **wbe.vector_points_to_raster**


**Delineate**

Watershed.delineateOutlets (many duplications as in Watershed.delineatePourPoints)

a. If wetlands exists\
    &nbsp; 1) BufferVector - Apply buffer to main stream\
    &nbsp; 2) Convert buffered vector to raster **Use Buffer Size**\
    &nbsp; 3) filter wetland - has been done before? **Here using the stream buffer**\ 
b. Add user defined outlets\
c. Add feedlot outlets\
d. Add dougout outlets\
d. Combine all outlets to a single raster\
e. If there is no wetland, feedlot or dugouts, get FlowD8. \
f. else burnFlowDirection\
e. FlowAccumD8\
f. ExtractStreams\
g. Wetland process - repeat step a?\
    &nbsp; 1) ExtractStreams\
    &nbsp; 2) RasterStreamToVector\
    &nbsp; 3) BufferVector - Apply buffer to main stream\
    &nbsp; 4) Convert buffered vector to raster\
    &nbsp; 5) filter wetland\
h. buildPourPoints\
i. repairSubbasinWithFlowDirChangeBMP - Apply wetland, feedlot and wetland to mask\
j. clipByMask\
k. Generate Subbasin with combined outlets - Plugin Watershed\
l. Generate streamnetwork to link all outlets - buildStreamNetworkLink2Outlets\
m. RasterStreamsToVector\
n. repairSubbasinWithFlowDirChangeBMP - Apply wetland, feedlot and wetland to subbasin raster\
o. reorderRasterID for subbasin raster\
p. reorderWetlandIDWithSubbasin\
q. WetlandSubbasinClassify\
r. buildWetlandRiparianInlet\
s. Generate slope - Slope\
t. Create slope percentage\
u. MaxUpslopeFlowpathLength\
v. Generate Stream order layer - StreamOrderImWEBs\
w. Create mask based on subbasin - GenerateMask\
x. Generate subbasin shapefile = RasterToVectorPolygon\
y. Dissolve subbasin based on id - SinglepartsToMultiparts\
z. Create shapefiles for other structure bmps.\






**Reclassify**

watershed.reclass just replace the raster value to the lookup table value and create a new raster

**Define Farm Field**

Waterhsed.defineFarmField

There are three options. Use Predefined only for now.
1. Automatic
2. Manual
3. Predefined. 

Steps:

1. filterAgriLanduse. If user select filter AgriLandUse, the the cells in Farm/Field where the landuse is not agriculture code will set to node data. The assumption is the farm/field will have the landuse code. It's better the Farm and Field is provied with shapefile?
2. Clip Farm and Field with mask
3. Convert Farm/Field to raster
4. Dissolve Field and Farm by ID - SinglepartsToMultiparts


**Calculate Watershed Parameters**

1. Waterhsed.buildWatershedParametersMaps
    a. Generate mask with subbasin,Landuse and Soil
    b. Calculate parameters with Customized Plugins - Done need some examples
        1) StreamWidth
        2) getReachDepthRaster
        3) ReclassLookup for fc1 and manning
        4) Velocity
        5) WetlandIndex
        6) SoilMoistureInitial
    c. Watershed.BuildReachParTable - Junzhi Students
    d. buildDSCRaster
    e. buildDSCAccumulateAvgMap
    f. buildCN2Raster
    g. buildCN2AccumulateAvgMap
    h. getFieldIDnAreas
    i. getFarmIDnAreas
    j. getSubbasinIDnAreas
    k. createSpatialTables
    l. buildBMPParMgtTables
    m. Build livestock parameter table
    n. Build manure_and_nutrient_parameter table
    o. Build LS_parameter table
    p. Build BMP index table
    q. Build subbasin multiplier table    r. 
2. Watersehd.createManureSetbackMap
    a.ExtractStreams
    b.ExcludeStreamInField
    c.AverageSlopeToStream
    d.createManureSetbackMap

In [ ]:
def getRasterCellAreaM2(raster:Raster):
    """Calculate the cell area in km2 assuming the resolution is in m"""
    return raster.configs.resolution_x * raster.configs.resolution_y

def getRasterCellAreaKM2(raster:Raster):
    """Calculate the cell area in km2 assuming the resolution is in m"""
    return getRasterCellAreaM2(raster) / 1e6
    

def calculateStreamDimension(flowAcc:Raster, mask:Raster, parameterA: float, parameterB: float) -> Raster:
    """
    
    Calculate Stream Width/Depth as (FlowAcc * Cell Area * A)^B. Width and depth will be calculated using different parameters. 
    
    Parameters:
    flowAcc         : Flow Accumualation Raster
    mask            : Stream Network Raster
    parameterA   : A
    parameterB   : B
    
    """

    return (flowAcc * getRasterCellArea(mask) * parameterA) ** parameterB

def lookupValues(inputRaster:Raster, mask: Raster, lookup:dict) -> Raster:
    """
    Replace the raster values based on lookup table. The application could be

    1. generate field capacity raster based on soil lookup table
    2. generate manning raster based on landuse lookup table
    3. 
    
    Parameters:
    inputRaster         : the input raster
    mask                : mask raster
    lookup              : lookup table assuming the input value is the key and output value is the value   
    """

    pass

def getSlopeRadius(slopeDeg:Raster)->Raster:
    return (slopeDeg * math.pi / 180).tan()

def calculateReachVelocity(manning:Raster, slopeRadius:Raster, streamDepth:Raster, mask:Raster, min:float = 0.005, max: float = 3) -> Raster:
    """
    Calculate full reach velocity using Manning equation

    Parameters:
    manning             : Manning raster
    slopeDeg            : Slope raster in degree. Could use radius instead
    streamDepth         : Stream depth raster
    mask                : mask raster
    min                 : min velocity
    max                 : max velocity

    """
    velocity = slopeRadius.sqrt() * streamDepth**0.6667 / manning
    return velocity.min(max).max(min)

def calculateWetnessIndex(flowAcc:Raster, slopeRadius:Raster, mask:Raster)->Raster:
    """
    Calculate wetness index using flow accumulation and slope
    """

    return (flowAcc * getRasterCellAreaM2(mask)).log() / slopeRadius.tan()

def calculateIntialSoilMoisture(wetnessIndex:Raster, fieldCapacity:Raster, mask:Raster, minSaturation:float=0.05, maxSaturation:float=1)->Raster:
    """
    Calculate the initial soil moisture using linear interpolation
    """

    minWetnessIndex = wetnessIndex.configs.minimum
    maxWetnessIndex = wetnessIndex.configs.maximum * 0.8

    wti = wetnessIndex.max(maxWetnessIndex)
    ratio = (wti - minWetnessIndex) * (maxSaturation - minSaturation) / (maxWetnessIndex - minWetnessIndex) + minSaturation
    return ratio * fieldCapacity

def buildReachParameterTable():
    pass

def calculatePotentialRunoffCoefficient(slope:Raster, mask:Raster, landuse:Raster, soil:Raster, reach:Raster, flowDirectionD8:Raster, reachWidth:dict)->Raster:
    
    #calculate reach water surface fraction
    #need to find a quick way to do lookup
    reachWaterSurfaceFraction = this.getReachWidth(reachID) / mask.configs.resolution_x    
    reachWaterSurfaceFraction = flowDirectionD8.con("(value == 1) || (value ==4) || (value == 8) || (value ==16)", 1.41421356 * reachWaterSurfaceFraction - 0.5 * reachWaterSurfaceFraction * reachWaterSurfaceFraction, reachWaterSurfaceFraction)

class LanuseParameters: 
    def __init__(self) -> None:
        self.dem = demName


def getPotentialRunoffCoefficient(landuseId:int, soilId:int, slope:float, reachFraction:float, landuseParameterLookup:pd.DataFrame, soilParameterLookup:pd.DataFrame):
    """
    
    Soid Id -> Soil Texture (1-12
    LanUse Id -> PRC_ST1, ..., PRC_ST12 where ST is for soil texture
    """





**Create imWEBs Model**

**Scenario Analysis**

1. Watershed.generateWeightFiles
2. Watershed.generateBMPDistFiles
3. Watershed.generateH5Java - The whole library IMWEBsHDF5Util needs to be converted - Looks like the library is created to fit dep data format. The engine HDF5 functions should be good enough if the rasters are in tiff format.  